In [13]:
FILE = 'scratch/Smaalenenes_1903_Excel_Correction.xlsx'   # <-- set to your Excel file name
OUT  = 'validation_report.csv'

In [14]:
import pandas as pd, re
from collections import Counter

df = pd.read_excel(FILE, dtype=str).fillna('').reset_index(drop=True)
cols = list(df.columns)
def has(*c): return all(x in cols for x in c)

issues, skipped = [], []
def flag(check, i, detail):
    r = df.iloc[i] if i is not None else {}
    issues.append({'check': check, 'excel_row': (i + 2) if i is not None else '',
                   'gaards_no': r.get('gaards_no', ''), 'brugs_no': r.get('brugs_no', ''),
                   'detail': detail})

mk = pd.to_numeric(df['mark'], errors='coerce').fillna(0) if 'mark' in cols else None
oe = pd.to_numeric(df['ore'],  errors='coerce').fillna(0) if 'ore'  in cols else None

if 'eier_bruker' in cols:
    eb = df['eier_bruker'].str.strip()
    is_total = eb.str.contains('overf', case=False, na=False)
    is_sogn  = eb.str.contains(r'\bsogn\b', case=False, na=False) & ~eb.str.contains(r'\d')
    is_data  = ~is_total & ~is_sogn
else:
    is_total = pd.Series(False, index=df.index)
    is_data  = pd.Series(True, index=df.index)

# type safety: numeric columns clean
num_re, ore_re = re.compile(r'^\d+$'), re.compile(r'^\d{1,2}$')
if has('mark', 'ore'):
    for i in df.index[is_data]:
        m, o = df.at[i, 'mark'].strip(), df.at[i, 'ore'].strip()
        if m and not num_re.match(m): flag('type', i, f'mark not numeric: {m!r}')
        if o and not ore_re.match(o): flag('type', i, f'ore not 2-digit: {o!r}')
        if m and not o: flag('type', i, 'mark without ore')
else: skipped.append('type (no mark/ore)')

# balance: independent page-sum recompute (ore units)
if has('mark', 'ore') and is_total.any():
    tot = list(df.index[is_total])
    for a, c in zip(tot, tot[1:]):
        sd = df.iloc[a + 1:c][is_data.iloc[a + 1:c].values]
        s = int(pd.to_numeric(sd['mark'], errors='coerce').fillna(0).sum()) * 100 + \
            int(pd.to_numeric(sd['ore'],  errors='coerce').fillna(0).sum())
        va = int(round(mk[a])) * 100 + int(round(oe[a])); vc = int(round(mk[c])) * 100 + int(round(oe[c]))
        if vc - va != s: flag('balance', c, f'delta {(vc - va - s)/100:.2f} (total {vc/100:.2f}, prev {va/100:.2f}, page {s/100:.2f})')
else: skipped.append('balance (no mark/ore or no totals)')

# words_in_CD: columns C and D should be numeric; flag stray words
LET = re.compile(r'[A-Za-zÆØÅæøå]')
for pos in (2, 3):
    if pos < len(cols):
        col = cols[pos]
        v = df.loc[is_data, col].str.strip(); ne = v[v != '']
        if len(ne) and ne.str.match(r'^\d+$').mean() > 0.5:
            for i in ne.index:
                if LET.search(df.at[i, col]): flag('words_in_CD', i, f'{col} (col {chr(65 + pos)}) has words: {df.at[i, col]!r}')

# brugs_no uniqueness within each gaards_no block (per herred)
if has('gaards_no', 'brugs_no'):
    if 'herred' in cols:
        h_col = df['herred'].fillna('').str.strip()
    else:
        h_col = pd.Series('', index=df.index)
        eb2 = df['eier_bruker'].str.strip() if 'eier_bruker' in cols else pd.Series('', index=df.index)
        cur_h = ''
        for i in df.index:
            e = eb2.iloc[i]
            if re.search(r'\bherred\b', e, re.I) and len(e) < 40 and not any(c.isdigit() for c in e):
                cur_h = e
            h_col.iloc[i] = cur_h
    s_col = df['sogn'].fillna('').str.strip() if 'sogn' in cols else pd.Series('', index=df.index)
    data = df[is_data].copy()
    data['_h'] = h_col[is_data]
    data['_s'] = s_col[is_data]
    data['_g'] = pd.to_numeric(data['gaards_no'], errors='coerce')
    data['_b'] = pd.to_numeric(data['brugs_no'], errors='coerce')
    data = data.dropna(subset=['_g', '_b'])
    dups = data[data.duplicated(subset=['_h', '_s', '_g', '_b'], keep=False)]
    for (h, s, g, b), grp in dups.groupby(['_h', '_s', '_g', '_b']):
        other_rows = [r + 2 for r in grp.index]
        for ix in grp.index:
            s = data.at[ix, '_s'] if '_s' in data.columns else ''
            gn = df.at[ix, 'gaardens_navn'] if 'gaardens_navn' in cols else ''
            ek = df.at[ix, 'eier_bruker'] if 'eier_bruker' in cols else ''
            others = [r for r in other_rows if r != ix + 2]
            flag('dup_brugs', ix, f'brugs_no={int(b)} gaards_no={int(g)} | {gn} | {ek} | also at row(s) {others} | {h or "?"}, {s or "?"}')
else: skipped.append('dup_brugs (missing columns)')

# herred/sogn overview
if 'herred' in cols:
    h_col = df['herred'].fillna('').str.strip()
    s_col = df['sogn'].fillna('').str.strip() if 'sogn' in cols else pd.Series('', index=df.index)
else:
    h_col, s_col = pd.Series('', index=df.index), pd.Series('', index=df.index)
    eb3 = df['eier_bruker'].str.strip() if 'eier_bruker' in cols else pd.Series('', index=df.index)
    ch3, cs3 = '', ''
    for i in df.index:
        e = eb3.iloc[i]
        if re.search(r'\bherred\b', e, re.I) and len(e) < 40 and not any(c.isdigit() for c in e): ch3 = e
        if re.search(r'\bsogn\b', e, re.I) and len(e) < 30 and not any(c.isdigit() for c in e): cs3 = e
        h_col.iloc[i] = ch3
        s_col.iloc[i] = cs3

ov_data = df[is_data].copy()
ov_data['_h'] = h_col[is_data].values
ov_data['_s'] = s_col[is_data].values
ov_data['_g'] = pd.to_numeric(ov_data['gaards_no'], errors='coerce')
ov_data['_m'] = pd.to_numeric(ov_data['mark'], errors='coerce').fillna(0) if 'mark' in cols else 0
ov_data['_o'] = pd.to_numeric(ov_data['ore'], errors='coerce').fillna(0) if 'ore' in cols else 0

h_order = list(dict.fromkeys(ov_data['_h']))
s_order = list(dict.fromkeys(zip(ov_data['_h'], ov_data['_s'])))
ov_data['_h'] = pd.Categorical(ov_data['_h'], categories=h_order, ordered=True)
ov = ov_data.groupby(['_h', '_s'], sort=False, observed=True).agg(
    gaards=('_g', 'nunique'),
    bruks=('_g', 'count'),
    total_mark=('_m', 'sum'),
    total_ore=('_o', 'sum'),
).reset_index()

for _, r in ov.iterrows():
    total_skyld = r['total_mark'] + r['total_ore'] / 100
    issues.append({'check': 'overview', 'excel_row': '',
                   'gaards_no': r['gaards'], 'brugs_no': r['bruks'],
                   'detail': f"{r['_h']} | {r['_s']} | {int(r['gaards'])} gaards, {int(r['bruks'])} bruks, skyld={total_skyld:.2f}"})

print(f"\nOverview: {ov['_h'].nunique()} herreds, {len(ov)} sogns")
for _, r in ov.iterrows():
    total_skyld = r['total_mark'] + r['total_ore'] / 100
    print(f"  {r['_h']:30s} {r['_s']:25s} {int(r['gaards']):>4d} gaards  {int(r['bruks']):>5d} bruks  skyld={total_skyld:>10.2f}")

rep = pd.DataFrame(issues, columns=['check', 'excel_row', 'gaards_no', 'brugs_no', 'detail'])
rep.to_csv(OUT, index=False, encoding='utf-8-sig')

print('columns:', cols)
if skipped: print('skipped checks:', skipped)
print(rep['check'].value_counts().to_string() if len(rep) else 'no issues')
print(f'\n{len(rep)} issues -> {OUT}')


Overview: 22 herreds, 40 sogns
  Trøgstad herred                Trøgstad sogn              102 gaards    109 bruks  skyld=   1559.39
  Trøgstad herred                Baastad sogn                78 gaards     82 bruks  skyld=    892.70
  Askim herred                   Askim Sogn                  98 gaards    107 bruks  skyld=   1440.64
  Spydeberg herred               Spydeberg sogn              57 gaards     59 bruks  skyld=    918.24
  Spydeberg herred               Hovin sogn                  60 gaards     61 bruks  skyld=    738.41
  Spydeberg herred               Heli sogn                   12 gaards     12 bruks  skyld=    152.09
  Skiptvet herred                Skiptvet sogn               87 gaards     97 bruks  skyld=   1178.57
  Rakkestad Herred               Rakkestad sogn             112 gaards    123 bruks  skyld=   1949.22
  Rakkestad Herred               Degernes sogn               83 gaards     89 bruks  skyld=    967.17
  Rakkestad Herred               Os sogn          